In [10]:
import pandas as pd
import json


"""Mount Google Drive
drive.mount('/content/gdrive', force_remount=True)

# Load JSON file
with open('gdrive/MyDrive/Data mining/Data mining/Data mining/data/allflight.json', 'r') as f:
    data = json.load(f)

with open('gdrive/MyDrive/Data mining/Data mining/Data mining/data/CodeContexte.json', 'r') as c:
    context_data = json.load(c)

# If JSON is a list of dictionaries
df = pd.DataFrame(data)
sumary_line

Keyword arguments:
argument -- description
Return: return_description
"""
filghts_path = 'data/All flights - data.csv'
# Save to CSV
data = pd.read_csv(filghts_path, index_col=False)
df = pd.DataFrame(data)


In [11]:
df.head()


,Operation time,Airline,Flight Num,Departure Airport,Arrival Airport,Terminal,Departure / Arrival,Origin Date,CodeContext
0,2021-12-29 16:49:00,AH,1011,ORY,ALG,4,Arrival,2021-12-29,ONB
1,2021-12-29 18:30:00,AH,2014,ALG,BCN,4,Departure,2021-12-29,OFB
2,2021-12-29 12:09:00,LH,1316,FRA,ALG,4,Arrival,2021-12-29,ONB
3,2021-12-29 13:25:00,LH,1317,ALG,FRA,4,Departure,2021-12-29,OFB
4,2021-12-29 10:03:00,5O,1051,CDG,ALG,4,Arrival,2021-12-29,ONB


In [12]:
df.tail()

,Operation time,Airline,Flight Num,Departure Airport,Arrival Airport,Terminal,Departure / Arrival,Origin Date,CodeContext
157713,2024-12-05 8:00:00,SF,3018,ALG,SXB,4,Departure,2024-12-05,SCT
157714,2024-12-05 9:00:00,SF,1850,ALG,CBH,1,Departure,2024-12-05,SCT
157715,2024-12-05 18:15:00,SF,1802,ALG,HME,1,Departure,2024-12-05,SCT
157716,2024-12-05 14:00:00,SF,1844,ALG,CZL,1,Departure,2024-12-05,SCT
157717,2024-12-05 8:40:00,SF,1848,ALG,BSK,1,Departure,2024-12-05,SCT


In [ ]:
# Rename columns to remove spaces and make them lowercase
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '')

# Now convert to datetime
df['operationtime'] = pd.to_datetime(df['operationtime'], errors='coerce')
df['origindate']    = pd.to_datetime(df['origindate'],    errors='coerce')

KeyError: 'operationtime'

In [ ]:
str_cols = ['airline', 'departureairport', 'arrivalairport', 'CodeContext']

for col in str_cols:
    df[col] = df[col].astype(str).str.strip().str.upper()
    df[col] = df[col].replace('', pd.NA)

In [ ]:
key_cols = ['flightnumber', 'operationtime', 'origindate','departureairport', 'arrivalairport', 'CodeContext']
df['missing_key_count'] = df[key_cols].isna().sum(axis=1)


In [ ]:
df_clean = df[df['missing_key_count'] <= 1].copy()

In [ ]:
mask_bad_fn = df_clean['flightnumber'].isna()
mask_bad_ap = df_clean['departureairport'].isna() & df_clean['arrivalairport'].isna()
df_clean = df_clean[~(mask_bad_fn | mask_bad_ap)].copy()

In [ ]:
df_clean = df_clean.drop(columns=['missing_key_count'])

In [ ]:
df_clean.to_csv('/content/gdrive/MyDrive/airport_data_clean.csv', index=False)

In [ ]:
print(df_clean.info())
print('-----------------------------------------------------')
print(df_clean.isna().sum())
print('-----------------------------------------------------')
print(df_clean[['airline', 'departureairport', 'arrivalairport', 'CodeContext']].head())


In [ ]:
# now we work with df_clean

In [ ]:
#Create new column For status_text (map codeContext => status_text)
df_clean['status_text'] = df_clean['CodeContext'].map(context_data)

df_clean.head()

In [ ]:
#check if mapping was not done somewhere
df_clean['status_text'].isna().sum()

In [ ]:
#Display values/counts per feature to detect noise / outliers
terminal_counts = df_clean['aircraftterminal'].value_counts(dropna=False)
print(terminal_counts)


In [ ]:
airline_counts = df_clean['airline'].value_counts(dropna=False)
print(airline_counts)
#2-letter codes are used

In [ ]:
status_counts = df_clean['status_text'].value_counts(dropna=False)
print(status_counts)

In [ ]:

distinct_arrivalairports = df_clean['arrivalairport'].unique()
print(distinct_arrivalairports.size)
print(distinct_arrivalairports)

arrivalairport_counts = df_clean['arrivalairport'].value_counts(dropna=False)
print(arrivalairport_counts)

In [ ]:
distinct_departureairport = df_clean['departureairport'].unique()
print(distinct_departureairport.size)
print(distinct_arrivalairports)

departureairport_counts = df_clean['departureairport'].value_counts(dropna=False)
print(departureairport_counts)

In [ ]:
# rest of preprocessing steps ..........

In [ ]:
# Visualization for the data :

In [ ]:
df_clean['date']      = df_clean['operationtime'].dt.date
df_clean['hour']      = df_clean['operationtime'].dt.hour
df_clean['weekday']   = df_clean['operationtime'].dt.weekday  # 0=Mon
df_clean['is_weekend'] = df_clean['weekday'].isin([5, 6])


In [ ]:
distinct_hours = df_clean['hour'].unique()
print(distinct_hours.size)
print(distinct_hours)

In [ ]:
df_clean['hour'].head(25)

In [ ]:
import matplotlib.pyplot as plt

df_clean['hour'].value_counts().sort_index().plot(kind='bar', figsize=(10,4))
plt.xlabel('Hour of day'); plt.ylabel('Number of flights'); plt.title('Flights per hour')
plt.show()


In [ ]:
# 0 is monday
df_clean['weekday'].value_counts().sort_index().plot(kind='bar', figsize=(10,4))
plt.xlabel('day of the week :'); plt.ylabel('Number of flights'); plt.title('Flights per weekday')
plt.show()

In [ ]:
df_clean.groupby('date').size().plot(figsize=(10,4))
plt.xlabel('Date'); plt.ylabel('Number of flights'); plt.title('Daily traffic')
plt.show()


In [ ]:
df_clean['status_text'].value_counts().plot(kind='bar', figsize=(8,4))
plt.title('Flight status distribution')
plt.show()
# we should check the status if they are repeated

In [ ]:
pd.crosstab(df_clean['hour'], df_clean['status_text']).plot(kind='bar', stacked=True, figsize=(12,4))
plt.title('Status by hour')
plt.show()

pd.crosstab(df_clean['aircraftterminal'], df_clean['status_text']).plot(kind='bar', stacked=True, figsize=(8,4))
plt.title('Status by terminal')
plt.show()


In [ ]:
df_clean['route'] = df_clean['departureairport'] + '-' + df_clean['arrivalairport']

In [ ]:
print(df_clean['route'].value_counts().head(10))
pd.crosstab(df_clean['route'], df_clean['status_text']).head(10)

In [ ]:
df_clean.tail(10)

In [ ]:
#Outliers handling ...

In [ ]:
#Terminal:
terminals_list = df['aircraftterminal'].astype(str).unique().tolist()
print(terminals_list)
valid_terminals = ['1', '2', '4']
invalid_values = [x for x in terminals_list if x not in valid_terminals]
print(invalid_values)

In [ ]:
#Visualization of terminals => Hajj / Omra
#filter flights where departure OR arrival is MED or JED
airports_of_interest = ['MED', 'JED']
filtered_df = df_clean[
    (df_clean['departureairport'].isin(airports_of_interest)) |
    (df_clean['arrivalairport'].isin(airports_of_interest))
]

terminal_counts = filtered_df['aircraftterminal'].value_counts()
print(terminal_counts)


In [ ]:
terminals = ['0', '3']
filtered_df = df_clean[
    (df_clean['aircraftterminal'].isin(terminals))
]
terminal_counts = filtered_df['route'].value_counts()
filtered_df.head(10)
print(terminal_counts)


In [ ]:
routes = ['MED-ALG','JED-ALG']
filtered_df = df_clean[
    (df_clean['route'].isin(routes))
]
terminal_counts = filtered_df['aircraftterminal'].value_counts()
filtered_df.head(10)
print(terminal_counts)


In [ ]:
import matplotlib.pyplot as plt

terminal_counts.plot(kind='bar', color='skyblue')
plt.title('Flights to/from MED or JED by Terminal')
plt.xlabel('Terminal')
plt.ylabel('Number of Flights')
plt.show()

In [ ]:
#Only for non hajj / omra
#Replace invalid terminal with valid terminal depending on simialr instances in terms of arrivalairport and departure airport
for idx, row in df_clean[df_clean['aircraftterminal'].isin(invalid_values)].iterrows():

    #Skip Hajj / Omra routes (ambiguous terminals)
    if (
        (not(row['departureairport'] in ['MED', 'JED'])) and
        (not(row['arrivalairport'] in ['MED', 'JED']))
    ):
      #find similar flights
      similar = df_clean[
          (df_clean['departureairport'] == row['departureairport']) &
          (df_clean['arrivalairport'] == row['arrivalairport']) &
          (~df_clean['aircraftterminal'].isin(invalid_values))
      ]

      #replace invalid by valid
      if not similar.empty:
          df_clean.at[idx, 'aircraftterminal'] = similar['aircraftterminal'].mode()[0]


In [ ]:
remaining_invalid = df_clean[
    (~df_clean['aircraftterminal'].isin(valid_terminals)) &
    (~df_clean['departureairport'].isin(['MED', 'JED'])) &
    (~df_clean['arrivalairport'].isin(['MED', 'JED']))
]

print("Remaining invalid terminals (non-Hajj/Omra):", len(remaining_invalid))
#could not be handled by previous algorithm

In [ ]:
#All assign terminal 4 (they are all international)
df_clean.loc[remaining_invalid.index, 'aircraftterminal'] = '4'


In [ ]:
check = df_clean[
    (~df_clean['aircraftterminal'].isin(valid_terminals)) &
    (~df_clean['departureairport'].isin(['MED', 'JED'])) &
    (~df_clean['arrivalairport'].isin(['MED', 'JED']))
]
print("Remaining invalid terminals (non-Hajj/Omra) CHECK:", len(check))
#could not be handled by previous algorithm

In [ ]:
#------------------------------------------------------
#All terminal outliers handled except for MED / JED ...
#------------------------------------------------------

In [ ]:
routes = ['MED-ALG','JED-ALG','ALG-JED','ALG-MED']
HajjUmrah = df_clean[(df_clean['route'].isin(routes))]
valid_terminals = ['1', '2', '4']
route_noise = df_clean[~df_clean['aircraftterminal'].isin(valid_terminals)]

In [ ]:
route_noise.head()

In [ ]:
df_clean.loc[route_noise.index, 'aircraftterminal'] = '2'

In [ ]:
df_clean['aircraftterminal'].value_counts()

In [ ]:

terminal_counts = df_clean['aircraftterminal'].value_counts()
terminal_counts.plot(kind='bar', color='skyblue')
plt.title('Flights by Terminal')
plt.xlabel('Terminal')
plt.ylabel('Number of Flights')
plt.show()

In [ ]:
flight_key = [
    "airline",
    "flightnumber",
    "origindate",
    "departureairport",
    "arrivalairport",
]

status_counts = (df_clean.groupby(flight_key)["status_text"].nunique()
      .reset_index(name="n_statuses"))
repeated_flights = status_counts[status_counts["n_statuses"] > 1]

print(repeated_flights.head())
print(repeated_flights.shape)


In [ ]:
dups = df_clean[df_clean.duplicated(subset=flight_key, keep=False)]
print(dups.head())
print(dups.shape)


In [ ]:
status_map = {
    "SCT": "Scheduled_OnTime",   # on time
    "EAR": "Scheduled_OnTime",   # early
    "DEL": "Delayed",            # delayed

    "BST": "Boarding",           # boarding
    "BEN": "Boarding",           # final boarding
    "GCL": "Boarding",           # gate closed
    "FCL": "Boarding",           # flight closed

    "OFB": "Departed",           # departed
    "THM": "EnRoute",            # in range
    "TEN": "EnRoute",            # approach

    "LAN": "Arrived",            # landed
    "ONB": "Arrived",            # arrived / on blocks

    "DX": "Cancelled",           # canceled

    "DV": "Diverted",            # diverted
    "RT": "Diverted",            # reroute
    "GRT": "Diverted",           # ground return
}


In [ ]:
df_clean['status_group'] = df_clean['CodeContext'].map(status_map)
unmapped = df_clean[df_clean['status_group'].isna()]['CodeContext'].unique()
print(unmapped)

In [ ]:
df_clean.to_csv('/content/gdrive/MyDrive/airport_data_clean.csv', index=False)